In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import ToTensor
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
class CustomWaferDataset(Dataset):
    def __init__(self, wafer_file):
        self.wafer_file = wafer_file
        self.defect_label = self.wafer_file['defect_label']
        self.x = torch.tensor(np.expand_dims(self.wafer_file['X'], axis = 1))
        self.x = self.x.float() / 2.0
        self.y = torch.tensor(self.wafer_file['y'], dtype=torch.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        wafer_map = self.x[idx]
        label = self.y[idx]
        return wafer_map, label

In [3]:
os.listdir()

['.ipynb_checkpoints',
 'datasets',
 'pytorch_testbase.ipynb',
 'Untitled.ipynb',
 'wafer_defect_detection.ipynb',
 'wafer_map_data.npz']

In [4]:
data = np.load('wafer_map_data.npz')

In [5]:
dataset = CustomWaferDataset(data)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, val_size])

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
device = torch.accelerator.current.accelerator().type if torch.accelerator.is_available() else "cpu"
print(f'Using device: {device}')

Using device: cpu


In [8]:
class ConvNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(8192, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
model = ConvNN()

# set loss function
criterion = nn.CrossEntropyLoss()

# set optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(2):
    running_loss = 0.0
    for i, (wafers, labels) in enumerate(train_dataloader):

        # zero the parameter gradients
        optimizer.zero_grad()

        # forwards pass
        outputs = model(wafers)
        loss = criterion(outputs, labels)

        # backward + optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 49:  # print every 100 mini-batch
            print(f'{[epoch + 1, i + 1]} loss: {running_loss / 100:.3f}')
            running_loss = 0.0
print('Finished Training')

[1, 50] loss: 0.369
[1, 150] loss: 0.169
[1, 250] loss: 0.133
[2, 50] loss: 0.038
[2, 150] loss: 0.098


In [21]:
# testing network
correct = 0
total = 0

with torch.no_grad():
    for wafers, labels in test_dataloader:
        outputs = model(wafers)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accurary of the network on the 10000 wafers: {100 * (correct / total)}%')

Accurary of the network on the 10000 wafers: 99.5%
